# Project 1: R&D Quintiles (CAPM + FF3)

Based on `recoded_project_0.ipynb`. **R&D:** No R&D vs. five **equally populated quintiles** of firms with XRD (option: **normalized** = XRD as % of net income, or **non-normalized** = raw XRD). **Models:** CAPM and Fama–French 3-factor. Same metrics; no investability flag.

In [248]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
import plotly.graph_objects as go
from pathlib import Path
import pickle

def connect_wharton(username="zkg232"):
    """Connect to WRDS PostgreSQL (our prior method: explicit username, uses .pgpass for password)."""
    return wrds.Connection(wrds_username=username)

db = connect_wharton()

# Cache for WRDS data to avoid re-running queries
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

def load_or_fetch(name, loader):
    path = CACHE_DIR / f"{name}.pkl"
    if path.exists():
        with open(path, 'rb') as f:
            return pickle.load(f)
    data = loader()
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    return data

Loading library list...
Done


## Compustat Query - Get R&D Data and Link to CRSP

In [249]:
comp_sql = """
SELECT
    a.gvkey, 
    a.datadate, 
    a.xrd,
    a.ni,
    a.revt,
    b.lpermno AS permno
FROM
    comp.funda a
JOIN
    crsp.ccmxpf_linktable b ON a.gvkey = b.gvkey
WHERE
    a.indfmt = 'INDL'
    AND a.datafmt = 'STD'
    AND a.popsrc = 'D'
    AND a.consol = 'C'
    AND a.fyear BETWEEN 1978 AND 2022
    AND a.fic = 'USA'
    AND a.curcd = 'USD'
    AND a.exchg BETWEEN 11 AND 19
    AND (a.sich < 6000 OR a.sich > 6999 OR a.sich IS NULL)
    AND (a.sich != 2834 OR a.sich IS NULL)
    AND b.linktype IN ('LU', 'LC')
    AND b.linkprim IN ('P', 'C')
    AND b.usedflag = 1
    AND (b.linkdt <= a.datadate OR b.linkdt IS NULL)
    AND (b.linkenddt >= a.datadate OR b.linkenddt IS NULL)
"""
comp = load_or_fetch('comp_quintiles_v2', lambda: db.raw_sql(comp_sql))

## CRSP Query - Monthly Returns with Delisting Adjustment

In [250]:
crsp_sql = """
SELECT 
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    c.dlret
FROM 
    crsp.msf a
INNER JOIN crsp.msenames b ON a.permno = b.permno
    AND a.date BETWEEN b.namedt AND b.nameendt
LEFT JOIN crsp.msedelist c ON a.permno = c.permno
    AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
WHERE 
    a.date BETWEEN '1970-01-01' AND '2023-06-30'
    AND a.ret IS NOT NULL
    AND a.ret >= -1
    AND b.shrcd IN (10, 11)
    AND (b.siccd < 6000 OR b.siccd > 6999 OR b.siccd IS NULL)
    AND (b.siccd != 2834 OR b.siccd IS NULL)
    AND b.exchcd IN (1, 2, 3)
"""
crsp = load_or_fetch('crsp', lambda: db.raw_sql(crsp_sql))

## Adjust for Delisting Returns

In [251]:
crsp['dlret'] = pd.to_numeric(crsp['dlret'], errors='coerce')
crsp['ret'] = np.where(crsp['dlret'].notna(),
                       (1 + crsp['ret']) * (1 + crsp['dlret']) - 1,
                       crsp['ret'])

## Market Return and Risk-Free Rate

In [252]:
mkt = load_or_fetch('mkt', lambda: db.raw_sql("""
    SELECT date, vwretd, ewretd
    FROM crsp.msi
    WHERE date BETWEEN '1970-01-01' AND '2023-06-30'
"""))

ff = load_or_fetch('ff_factors', lambda: db.raw_sql("""
    SELECT date, rf, mktrf, smb, hml
    FROM ff.factors_monthly
    WHERE date BETWEEN '1970-01-01' AND '2023-06-30'
"""))

## R&D Measure: Lag and (Option) Normalize

In [253]:
comp = comp.assign(
    xrd=pd.to_numeric(comp['xrd'], errors='coerce'),
    ni=pd.to_numeric(comp['ni'], errors='coerce'),
    revt=pd.to_numeric(comp['revt'], errors='coerce')
).sort_values(['permno', 'datadate'])
for lag in range(1, 5):
    comp[f'xrd_lag{lag}'] = comp.groupby('permno')['xrd'].shift(lag)
comp['xrdw'] = (comp['xrd'].fillna(0) + 0.8*comp['xrd_lag1'].fillna(0) + 0.6*comp['xrd_lag2'].fillna(0)
                + 0.4*comp['xrd_lag3'].fillna(0) + 0.2*comp['xrd_lag4'].fillna(0))
comp['xrdw_pct_ni'] = np.where(comp['ni'].notna() & (comp['ni'] > 0), comp['xrdw'] / comp['ni'], np.nan)
comp['xrdw_pct_revt'] = np.where(comp['revt'].notna() & (comp['revt'] > 0), comp['xrdw'] / comp['revt'], np.nan)
comp['datadate'] = pd.to_datetime(comp['datadate'])
comp['available_date'] = (comp['datadate'] + pd.Timedelta(days=120)).dt.to_period('M').dt.to_timestamp()
comp = comp.drop_duplicates(subset=['permno', 'datadate'], keep='last')
crsp['date'] = pd.to_datetime(crsp['date']).dt.to_period('M').dt.to_timestamp()

## 4-month (120-day) reporting lag — available_date

In [254]:
pass

## Market Cap - Use Lagged for Value Weighting

In [255]:
crsp = crsp.assign(
    ret=pd.to_numeric(crsp['ret'], errors='coerce'),
    prc=pd.to_numeric(crsp['prc'], errors='coerce'),
    shrout=pd.to_numeric(crsp['shrout'], errors='coerce')
).assign(mktcap=lambda x: x['prc'].abs() * x['shrout']).sort_values(['permno', 'date'])
crsp['mktcap_lag'] = crsp.groupby('permno')['mktcap'].shift(1)

## Merge Datasets (monthly formation: reprex-style merge + date filter + keep last)

In [256]:
comp_merge = (comp[['permno', 'available_date', 'xrdw', 'ni', 'xrdw_pct_ni', 'revt', 'xrdw_pct_revt']]
    .assign(permno=lambda d: pd.to_numeric(d['permno'], errors='coerce').astype('int64'))
    .dropna(subset=['permno']))
crsp_dates = crsp[['permno', 'date']].dropna().drop_duplicates()
crsp_dates['permno'] = pd.to_numeric(crsp_dates['permno'], errors='coerce').astype('int64')

merged = (crsp_dates.merge(comp_merge, on='permno', how='inner')
    .query('date >= available_date')
    .sort_values(['permno', 'date', 'available_date'])
    .drop_duplicates(subset=['permno', 'date'], keep='last')
    .merge(crsp[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag']], on=['permno', 'date'], how='inner'))
merged = merged[(merged['available_date'].dt.year >= 1980) & (merged['available_date'].dt.year <= 2022)]
merged = merged.dropna(subset=['ret', 'mktcap_lag']).query('mktcap_lag > 0')
merged['xrdw_pct_mkteq'] = np.where(merged['mktcap'].notna() & (merged['mktcap'] > 0), merged['xrdw'] / merged['mktcap'], np.nan)

## Analysis Data: R&D Quintiles (No R&D + Q1–Q5, equal count)

In [257]:

def assign_quintiles_per_date(df):
    out = []
    for date, g in df.groupby('date'):
        no_rd = g[~g['has_rd']].copy()
        no_rd['rd_quintile'] = 'No R&D'
        with_rd = g[g['has_rd']].copy()
        if len(with_rd) < 5:
            with_rd['rd_quintile'] = 'Q1'
        else:
            with_rd['rd_quintile'] = pd.qcut(with_rd['rd_measure'].rank(method='first'), 5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5']).astype(str)
        out.append(pd.concat([no_rd, with_rd], ignore_index=True))
    return pd.concat(out, ignore_index=True)

def build_analysis_data(merged, norm_metric):
    """norm_metric: None = raw XRDW, 'ni' = XRDW % NI, 'revt' = XRDW % Revenue, 'mkteq' = XRDW % Market Equity."""
    ad = merged[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag', 'xrdw', 'ni', 'xrdw_pct_ni', 'revt', 'xrdw_pct_revt', 'xrdw_pct_mkteq']].copy()
    if norm_metric is None:
        ad['rd_measure'] = ad['xrdw']
    elif norm_metric == 'ni':
        ad['rd_measure'] = ad['xrdw_pct_ni']
    elif norm_metric == 'revt':
        ad['rd_measure'] = ad['xrdw_pct_revt']
    elif norm_metric == 'mkteq':
        ad['rd_measure'] = ad['xrdw_pct_mkteq']
    else:
        raise ValueError(f"norm_metric must be None, 'ni', 'revt', or 'mkteq'; got {norm_metric}")
    ad['has_rd'] = ad['rd_measure'].notna() & (ad['rd_measure'] > 0)
    ad = assign_quintiles_per_date(ad.sort_values(['date', 'permno']))
    ad['portfolio_id'] = ad['rd_quintile']
    return ad

analysis_data_norm = build_analysis_data(merged, 'ni')
analysis_data_nonnorm = build_analysis_data(merged, None)
analysis_data_revt = build_analysis_data(merged, 'revt')
analysis_data_mkteq = build_analysis_data(merged, 'mkteq')


## Portfolio Returns, CAPM, FF3, Tables, Chart

In [258]:

def format_numeric_cols(data, rounding_map):
    data = data.copy()
    for col, digits in rounding_map.items():
        if col in data.columns and pd.api.types.is_numeric_dtype(data[col]):
            data[col] = data[col].round(digits)
    return data

def calculate_portfolio_returns(data, group_var='research_not'):
    ret_col = 'ret' if 'ret' in data.columns else 'ret_crsp'
    prc_col = 'prc' if 'prc' in data.columns else 'prc_crsp'
    data = data[data[ret_col].notna() & data[prc_col].notna() & data['mktcap'].notna()].copy()
    data = data.sort_values(['permno', 'date'])
    if 'mktcap_lag' not in data.columns:
        data['mktcap_lag'] = data.groupby('permno')['mktcap'].shift(1)
    data['prc_lag'] = data.groupby('permno')[prc_col].shift(1)
    data = data[data['mktcap_lag'].notna() & data['prc_lag'].notna()]
    data['date'] = data['date'] + pd.offsets.MonthEnd(0)
    def calc_vw_return(group):
        total_mktcap = group['mktcap_lag'].sum()
        if total_mktcap == 0:
            return np.nan
        weights = group['mktcap_lag'] / total_mktcap
        return (weights * group[ret_col]).sum()
    portfolio_returns = data.groupby(['date', group_var]).apply(
        lambda x: pd.Series({
            'vw_return': calc_vw_return(x),
            'ew_return': x[ret_col].mean(),
            'total_mktcap': x['mktcap_lag'].sum(),
            'n_stocks': len(x)
        })
    ).reset_index()
    return portfolio_returns.sort_values([group_var, 'date'])

def add_long_short_spreads(portfolio_returns, group_col='portfolio_id'):
    if group_col not in portfolio_returns.columns or 'ew_return' not in portfolio_returns.columns:
        return portfolio_returns
    wide_ew = portfolio_returns.pivot(index='date', columns=group_col, values='ew_return')
    wide_vw = portfolio_returns.pivot(index='date', columns=group_col, values='vw_return')
    needed = ['No R&D', 'Q1', 'Q5']
    for col in needed:
        if col not in wide_ew.columns or col not in wide_vw.columns:
            return portfolio_returns
    out = [portfolio_returns]
    for name, minus_col in [('Q5-No R&D', 'No R&D'), ('Q5-Q1', 'Q1')]:
        spread_ew = (wide_ew['Q5'] - wide_ew[minus_col]).dropna()
        spread_vw = (wide_vw['Q5'] - wide_vw[minus_col]).dropna()
        common_idx = spread_ew.index.intersection(spread_vw.index)
        spread_ew, spread_vw = spread_ew.loc[common_idx], spread_vw.loc[common_idx]
        spread_df = pd.DataFrame({
            'date': spread_ew.index,
            group_col: name,
            'ew_return': spread_ew.values,
            'vw_return': spread_vw.values,
            'total_mktcap': 0.0,
            'n_stocks': 0
        })
        out.append(spread_df)
    return pd.concat(out, ignore_index=True).sort_values([group_col, 'date'])

def run_capm_regression(portfolio_returns, market_returns, rf_rate, use_ew_benchmark=False):
    if 'rf' not in market_returns.columns or market_returns['rf'].isna().all():
        raise ValueError("Risk-free rate 'rf' must be present in market_returns (from Fama-French ff.factors_monthly).")
    market_col = 'ewretd' if use_ew_benchmark else 'vwretd'
    portfolio_returns = portfolio_returns.copy()
    portfolio_returns['date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
    cols = ['date', market_col, 'rf']
    market_data = market_returns[cols].copy()
    market_data['date'] = market_data['date'] + pd.offsets.MonthEnd(0)
    market_data = market_data.rename(columns={market_col: 'market_return'})
    reg_data = portfolio_returns.merge(market_data, on='date', how='left')
    reg_data = reg_data[reg_data['portfolio_return'].notna() & reg_data['market_return'].notna() & reg_data['rf'].notna()].copy()
    r = reg_data['rf'].astype(float)
    reg_data['portfolio_excess'] = reg_data['portfolio_return'] - r
    reg_data['market_excess'] = reg_data['market_return'] - r
    reg_data = reg_data[reg_data['portfolio_excess'].notna() & reg_data['market_excess'].notna()].copy()
    if len(reg_data) < 2:
        return None
    market_excess = pd.to_numeric(reg_data['market_excess'], errors='coerce')
    portfolio_excess = pd.to_numeric(reg_data['portfolio_excess'], errors='coerce')
    valid_mask = market_excess.notna() & portfolio_excess.notna()
    market_excess = market_excess[valid_mask].to_numpy(dtype=float)
    portfolio_excess = portfolio_excess[valid_mask].to_numpy(dtype=float)
    if len(market_excess) < 2:
        return None
    X = pd.DataFrame({'market_excess': market_excess})
    X = add_constant(X, has_constant='add')
    model = OLS(portfolio_excess, X).fit()
    n_months = len(portfolio_excess)
    arithmetic_monthly_return = pd.to_numeric(reg_data.loc[valid_mask, 'portfolio_return'], errors='coerce').mean()
    monthly_vol = portfolio_excess.std(ddof=1)
    rf_mean_sample = reg_data.loc[valid_mask, 'rf'].mean()
    params, tvalues, pvalues = model.params, model.tvalues, model.pvalues
    alpha = params.iloc[0]
    beta = params.iloc[1]
    alpha_tstat = tvalues.iloc[0]
    beta_tstat = tvalues.iloc[1]
    alpha_pval = pvalues.iloc[0]
    beta_pval = pvalues.iloc[1]
    ci = model.conf_int(alpha=0.05)
    return pd.Series({
        'arithmetic_monthly_return': arithmetic_monthly_return,
        'annualized_return': (1 + arithmetic_monthly_return) ** 12 - 1,
        'alpha': alpha, 'beta': beta,
        'alpha_tstat': alpha_tstat, 'beta_tstat': beta_tstat,
        'alpha_pval': alpha_pval, 'beta_pval': beta_pval,
        'alpha_lower_ci': ci.iloc[0, 0], 'alpha_upper_ci': ci.iloc[0, 1],
        'beta_lower_ci': ci.iloc[1, 0], 'beta_upper_ci': ci.iloc[1, 1],
        'r_squared': model.rsquared,
        'sharpe_ratio': (arithmetic_monthly_return - rf_mean_sample) / monthly_vol * np.sqrt(12) if monthly_vol > 0 else np.nan,
        'annualized_alpha': alpha * 12,
        'monthly_vol': monthly_vol,
        'annualized_vol': monthly_vol * np.sqrt(12),
        'n_observations': n_months
    })

def run_ff3_regression(portfolio_returns, market_returns, rf_rate):
    """Fama-French 3-factor: R - rf = alpha + b_mkt*mktrf + b_smb*smb + b_hml*hml."""
    for c in ['rf', 'mktrf', 'smb', 'hml']:
        if c not in market_returns.columns or market_returns[c].isna().all():
            raise ValueError(f"FF3 factor '{c}' must be in market_returns (from ff.factors_monthly).")
    portfolio_returns = portfolio_returns.copy()
    portfolio_returns['date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
    cols = ['date', 'mktrf', 'smb', 'hml', 'rf']
    market_data = market_returns[cols].copy()
    market_data['date'] = pd.to_datetime(market_data['date']) + pd.offsets.MonthEnd(0)
    reg_data = portfolio_returns.merge(market_data, on='date', how='left')
    reg_data = reg_data[reg_data['portfolio_return'].notna() & reg_data['mktrf'].notna() & reg_data['rf'].notna()].copy()
    reg_data['portfolio_excess'] = reg_data['portfolio_return'] - reg_data['rf'].astype(float)
    reg_data = reg_data[reg_data['portfolio_excess'].notna()].copy()
    if len(reg_data) < 5:
        return None
    y = reg_data['portfolio_excess'].astype(float).values
    X = reg_data[['mktrf', 'smb', 'hml']].astype(float)
    X = add_constant(X, has_constant='add')
    model = OLS(y, X).fit()
    n_months = len(y)
    arithmetic_monthly_return = reg_data['portfolio_return'].mean()
    monthly_vol = reg_data['portfolio_excess'].std(ddof=1)
    rf_mean = reg_data['rf'].mean()
    params, tvalues, pvalues, ci = model.params, model.tvalues, model.pvalues, model.conf_int(alpha=0.05)
    return pd.Series({
        'arithmetic_monthly_return': arithmetic_monthly_return,
        'annualized_return': (1 + arithmetic_monthly_return) ** 12 - 1,
        'alpha': params.iloc[0], 'beta': params.iloc[1],
        'beta_mkt': params.iloc[1], 'beta_smb': params.iloc[2], 'beta_hml': params.iloc[3],
        'alpha_tstat': tvalues.iloc[0], 'beta_tstat': tvalues.iloc[1],
        'beta_smb_tstat': tvalues.iloc[2], 'beta_hml_tstat': tvalues.iloc[3],
        'alpha_pval': pvalues.iloc[0], 'beta_pval': pvalues.iloc[1],
        'alpha_lower_ci': ci.iloc[0, 0], 'alpha_upper_ci': ci.iloc[0, 1],
        'beta_lower_ci': ci.iloc[1, 0], 'beta_upper_ci': ci.iloc[1, 1],
        'r_squared': model.rsquared,
        'sharpe_ratio': (arithmetic_monthly_return - rf_mean) / monthly_vol * np.sqrt(12) if monthly_vol > 0 else np.nan,
        'n_observations': n_months
    })

def calculate_sharpe_ratio(returns, rf_rate):
    if len(returns) == 0 or returns.isna().all():
        return np.nan
    excess = returns - rf_rate
    sd = returns.std()
    if sd == 0 or pd.isna(sd):
        return np.nan
    return (excess.mean() / sd) * np.sqrt(12)

def calculate_all_portfolio_metrics(portfolio_data, market_data, portfolio_name, rf_rate, model='capm'):
    """model: 'capm' or 'ff3'."""
    group_col = portfolio_data.columns[1]
    run_reg = (lambda pr, md, rf: run_capm_regression(pr, md, rf, use_ew_benchmark=False)) if model == 'capm' else run_ff3_regression
    ew_returns = portfolio_data[['date', 'ew_return', group_col]].copy()
    ew_returns = ew_returns.rename(columns={'ew_return': 'portfolio_return'})
    ew_metrics_list = []
    for group_name, group_data in ew_returns.groupby(group_col):
        r = run_reg(group_data[['date', 'portfolio_return']], market_data, rf_rate)
        if r is not None and len(r) > 0:
            ew_metrics_list.append({**r.to_dict(), group_col: group_name})
    ew_metrics = pd.DataFrame(ew_metrics_list)
    if len(ew_metrics) > 0:
        ew_metrics['weighting'] = 'Equal-Weighted'
    vw_returns = portfolio_data[['date', 'vw_return', group_col]].copy()
    vw_returns = vw_returns.rename(columns={'vw_return': 'portfolio_return'})
    vw_metrics_list = []
    for group_name, group_data in vw_returns.groupby(group_col):
        r = run_reg(group_data[['date', 'portfolio_return']], market_data, rf_rate)
        if r is not None and len(r) > 0:
            vw_metrics_list.append({**r.to_dict(), group_col: group_name})
    vw_metrics = pd.DataFrame(vw_metrics_list)
    if len(vw_metrics) > 0:
        vw_metrics['weighting'] = 'Value-Weighted'
    if len(ew_metrics) > 0 and len(vw_metrics) > 0:
        all_metrics = pd.concat([ew_metrics, vw_metrics], ignore_index=True)
    elif len(ew_metrics) > 0:
        all_metrics = ew_metrics
    elif len(vw_metrics) > 0:
        all_metrics = vw_metrics
    else:
        return pd.DataFrame()
    all_metrics['portfolio'] = all_metrics['weighting'] + '_' + all_metrics[group_col].astype(str)
    all_metrics = all_metrics.drop(columns=[group_col])
    return all_metrics

def format_performance_table(metrics, model='capm'):
    if metrics is None or (isinstance(metrics, pd.DataFrame) and (metrics.empty or 'portfolio' not in metrics.columns)):
        return metrics if isinstance(metrics, pd.DataFrame) else pd.DataFrame()
    rounding_map = {
        'arithmetic_monthly_return': 6, 'annualized_return': 4, 'alpha': 6, 'beta': 6,
        'alpha_tstat': 6, 'beta_tstat': 2, 'alpha_lower_ci': 6, 'alpha_upper_ci': 6,
        'beta_lower_ci': 6, 'beta_upper_ci': 6, 'r_squared': 6, 'sharpe_ratio': 6,
        'monthly_vol': 6, 'annualized_vol': 6,
        'beta_mkt': 6, 'beta_smb': 6, 'beta_hml': 6, 'beta_smb_tstat': 2, 'beta_hml_tstat': 2
    }
    table = metrics.copy()
    if 'portfolio' not in table.columns and 'weighting' in table.columns:
        group_col = [c for c in ['portfolio_id', 'research_not'] if c in table.columns]
        if group_col:
            table['portfolio'] = table['weighting'] + '_' + table[group_col[0]].astype(str)
    table['Portfolio'] = table['portfolio']
    table = format_numeric_cols(table, rounding_map)
    table['Alpha p-value'] = table['alpha_pval'].apply(lambda x: f"{x:.3e}")
    table['Beta p-value'] = table['beta_pval'].apply(lambda x: f"{x:.3e}")
    base_cols = ['Portfolio', 'arithmetic_monthly_return', 'monthly_vol', 'annualized_return', 'alpha', 'beta', 'alpha_tstat', 'beta_tstat', 'Alpha p-value', 'Beta p-value', 'alpha_lower_ci', 'alpha_upper_ci', 'beta_lower_ci', 'beta_upper_ci', 'r_squared', 'sharpe_ratio']
    renames = {'arithmetic_monthly_return': 'Mean Return (monthly)', 'monthly_vol': 'Std Return (monthly)', 'annualized_return': 'Annualized Return', 'alpha': 'Alpha (Monthly)', 'beta': 'Beta (Mkt)', 'alpha_tstat': 'Alpha t-stat', 'beta_tstat': 'Beta t-stat', 'alpha_lower_ci': 'Alpha Lower CI (95%)', 'alpha_upper_ci': 'Alpha Upper CI (95%)', 'beta_lower_ci': 'Beta Lower CI (95%)', 'beta_upper_ci': 'Beta Upper CI (95%)', 'r_squared': 'R-squared', 'sharpe_ratio': 'Sharpe Ratio'}
    if model == 'ff3' and 'beta_smb' in table.columns:
        base_cols = base_cols[:5] + ['beta_smb', 'beta_hml', 'beta_smb_tstat', 'beta_hml_tstat'] + base_cols[5:]
        renames['beta_smb'], renames['beta_hml'] = 'SMB Beta', 'HML Beta'
        renames['beta_smb_tstat'], renames['beta_hml_tstat'] = 'SMB t-stat', 'HML t-stat'
    out = table[[c for c in base_cols if c in table.columns]].copy()
    return out.rename(columns=renames)

def create_alpha_table(portfolio_metrics, market_returns, rf_rate_monthly):
    benchmark_sharpe_vw = calculate_sharpe_ratio(market_returns['vwretd'], rf_rate_monthly)
    benchmark_rows = pd.DataFrame({
        'Portfolio': ['CRSP VW Benchmark'],
        'Alpha (monthly %)': ['—'], 't-statistic': ['—'], 'p-value': ['—'],
        'Beta': ['—'], 'R²': ['—'],
        'Sharpe': [round(benchmark_sharpe_vw, 2)],
        'is_significant': [False], 'sort_order': [99]
    })
    def portfolio_display_name(x):
        # portfolio is e.g. 'Value-Weighted_No R&D', 'Equal-Weighted_Q3'
        parts = x.split('_', 1)
        if len(parts) != 2:
            return x
        wname = 'VW' if 'Value' in parts[0] else 'EW'
        return f"{parts[1]} {wname}"
    alpha_table = portfolio_metrics.copy()
    alpha_table['Portfolio'] = alpha_table['portfolio'].apply(portfolio_display_name)
    alpha_table['alpha_monthly_pct'] = (alpha_table['alpha'] * 100).round(2)
    alpha_table['alpha_star'] = alpha_table['alpha_pval'].apply(lambda x: '***' if x < 0.01 else '**' if x < 0.05 else '*' if x < 0.10 else '')
    alpha_table['Alpha (monthly %)'] = alpha_table['alpha_monthly_pct'].apply(lambda x: f"{x:.2f}") + alpha_table['alpha_star']
    alpha_table['t-statistic'] = alpha_table['alpha_tstat'].round(2)
    alpha_table['p-value'] = alpha_table['alpha_pval'].apply(lambda x: f"{x:.3f}")
    alpha_table['Beta'] = alpha_table['beta'].round(2)
    is_ff3 = 'beta_smb' in alpha_table.columns and 'beta_hml' in alpha_table.columns
    if is_ff3:
        alpha_table['Beta (Mkt)'] = alpha_table['beta'].round(2)
        alpha_table['Beta (SMB)'] = alpha_table['beta_smb'].round(2)
        alpha_table['Beta (HML)'] = alpha_table['beta_hml'].round(2)
    alpha_table['R²'] = alpha_table['r_squared'].round(2)
    alpha_table['Sharpe'] = alpha_table['sharpe_ratio'].round(2)
    alpha_table['is_significant'] = alpha_table['alpha_pval'] < 0.10
    sort_map = {'No R&D VW': 0, 'Q1 VW': 1, 'Q2 VW': 2, 'Q3 VW': 3, 'Q4 VW': 4, 'Q5 VW': 5, 'Q5-No R&D VW': 5.3, 'Q5-Q1 VW': 5.5,
                'No R&D EW': 6, 'Q1 EW': 7, 'Q2 EW': 8, 'Q3 EW': 9, 'Q4 EW': 10, 'Q5 EW': 11, 'Q5-No R&D EW': 11.3, 'Q5-Q1 EW': 11.5}
    alpha_table['sort_order'] = alpha_table['Portfolio'].map(sort_map).fillna(9)
    alpha_table = alpha_table.sort_values('sort_order')
    display_cols = ['Portfolio', 'Alpha (monthly %)', 't-statistic', 'p-value']
    if is_ff3:
        display_cols += ['Beta (Mkt)', 'Beta (SMB)', 'Beta (HML)']
    else:
        display_cols += ['Beta']
    display_cols += ['R²', 'Sharpe', 'is_significant', 'sort_order']
    alpha_table = alpha_table[[c for c in display_cols if c in alpha_table.columns]].astype(str)
    if is_ff3:
        benchmark_rows = pd.DataFrame({
            'Portfolio': ['CRSP VW Benchmark'], 'Alpha (monthly %)': ['—'], 't-statistic': ['—'], 'p-value': ['—'],
            'Beta (Mkt)': ['—'], 'Beta (SMB)': ['—'], 'Beta (HML)': ['—'], 'R²': ['—'],
            'Sharpe': [round(benchmark_sharpe_vw, 2)], 'is_significant': [False], 'sort_order': [99]
        })
    benchmark_rows = benchmark_rows.astype(str)
    alpha_table = pd.concat([alpha_table, benchmark_rows], ignore_index=True)
    alpha_table = alpha_table.sort_values('sort_order').drop(columns='sort_order')
    return alpha_table

def create_alpha_table_html(alpha_table, custom_css, caption_suffix='', table_type=""):
    """Create HTML alpha table with same styling (color-coded rows). Columns taken from alpha_table."""
    alpha_table_display = alpha_table.drop(columns='is_significant', errors='ignore')
    display_cols = [c for c in alpha_table_display.columns]
    if table_type == 'FF3':
        caption_title = "Alpha Estimation Results: Three-Factor Model"
        caption_eq = "Rp - Rƒ = α + β_mkt(MKT-Rƒ) + β_smb SMB + β_hml HML"
    else:
        caption_title = "Alpha Estimation Results: Single-Factor Model"
        caption_eq = "Rp - Rƒ = α + β(R_CRSP_VW - Rƒ)"
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
    {custom_css}
    </head>
    <body>
    <table class="table table-dark table-striped">
    <caption><b>{caption_title}</b><br>
    <i>{caption_eq}</i>{caption_suffix}<br>
    <i>{table_type}</i></caption>
    <thead>
    <tr>
    """ + "".join(f"<th>{c}</th>" for c in display_cols) + """
    </tr>
    </thead>
    <tbody>
    """
    for _, row in alpha_table_display.iterrows():
        portfolio = row['Portfolio']
        if portfolio == 'CRSP VW Benchmark':
            color = 'white'
        elif 'No R&D' in portfolio:
            color = '#003f5c'
        elif 'Q5-No R&D' in portfolio: color = '#d80020'
        elif 'Q5-Q1' in portfolio: color = '#00c9a7'
        elif 'Q1' in portfolio: color = '#ffa600'
        elif 'Q2' in portfolio: color = '#dd5182'
        elif 'Q3' in portfolio: color = '#955196'
        elif 'Q4' in portfolio: color = '#bc5090'
        elif 'Q5' in portfolio: color = '#ff6361'
        else:
            color = 'white'
        bold = 'font-weight: bold;' if portfolio == 'CRSP VW Benchmark' or 'No R&D' in portfolio or 'Q5-Q1' in portfolio or any(f'Q{i}' in portfolio for i in range(1,6)) else ''
        cells = "".join(f"<td>{row[c]}</td>" for c in display_cols)
        html += f"""
        <tr style="color: {color}; {bold}">
        {cells}
        </tr>
        """
    html += """
    </tbody>
    </table>
    </body>
    </html>
    """
    return html

custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Tomorrow:wght@400;600;700&display=swap');
body { font-family: 'Tomorrow', sans-serif; background-color: #1d1d1c; padding: 1rem; margin: 0; color: white; }
.table-dark.table-striped tbody tr:nth-of-type(odd) { background-color: rgba(255, 255, 255, 0.05) !important; }
.table-dark.table-striped tbody tr:nth-of-type(even) { background-color: rgba(0, 0, 0, 0.15) !important; }
.table-dark tbody tr[style*='color'] { background-color: inherit !important; }
.table-dark td, .table-dark th { padding: 0.5rem 0.75rem !important; }
</style>
"""

def create_performance_chart(chart_data, panel_type='vw', colors=None):
    pattern = 'vw_' if panel_type == 'vw' else 'ew_'
    benchmark_name = 'CRSP VW Benchmark'
    panel_data = chart_data[chart_data['key'].str.contains(pattern, na=False)].copy()
    def key_to_name(x):
        if 'CRSP' in str(x): return benchmark_name
        if '_' in str(x): return str(x).split('_', 1)[-1]
        return x
    panel_data['portfolio_name'] = panel_data['key'].apply(key_to_name)
    fig = go.Figure()
    for portfolio in panel_data['portfolio_name'].unique():
        port_data = panel_data[panel_data['portfolio_name'] == portfolio].sort_values('date')
        color = (colors or {}).get(portfolio, '#888')
        fig.add_trace(go.Scatter(x=port_data['date'], y=port_data['cumvalue'], mode='lines', name=portfolio, line=dict(color=color, width=3)))
    fig.update_layout(title='Cumulative Performance (1980-2022)', xaxis_title='Date', yaxis_title='Cumulative Value ($)',
                      yaxis=dict(rangemode='tozero', tickformat='$.2f'), template='plotly_dark', height=500)
    return fig


## Build Market Returns and Portfolio Metrics

In [259]:
mkt['date'] = pd.to_datetime(mkt['date']).dt.to_period('M').dt.to_timestamp()
ff['date'] = pd.to_datetime(ff['date']).dt.to_period('M').dt.to_timestamp()
for c in ['rf', 'mktrf', 'smb', 'hml']:
    ff[c] = pd.to_numeric(ff[c], errors='coerce')
market_returns = mkt.merge(ff, on='date', how='left')
if 'rf' not in market_returns.columns or market_returns['rf'].isna().all():
    raise ValueError("Risk-free rate 'rf' must be present (from ff.factors_monthly).")
market_returns['vwretd'] = pd.to_numeric(market_returns['vwretd'], errors='coerce')
market_returns['ewretd'] = pd.to_numeric(market_returns['ewretd'], errors='coerce')

rf_rate_monthly = market_returns['rf'].mean()

def run_quintile_metrics(analysis_data, market_returns, rf_rate_monthly):
    """Portfolio returns (with Q5-Q1 spread) and CAPM/FF3 metrics for one analysis_data."""
    pr = calculate_portfolio_returns(analysis_data, 'portfolio_id')
    pr = add_long_short_spreads(pr, 'portfolio_id')
    pm_capm = calculate_all_portfolio_metrics(pr, market_returns, 'R&D Quintiles', rf_rate_monthly, model='capm')
    pm_ff3 = calculate_all_portfolio_metrics(pr, market_returns, 'R&D Quintiles', rf_rate_monthly, model='ff3')
    return pr, pm_capm, pm_ff3

def metrics_for_period(portfolio_returns, market_returns, rf_rate_monthly, start_date, end_date):
    """Compute CAPM and FF3 metrics for a subperiod (start_date, end_date inclusive)."""
    start_ts = pd.Timestamp(start_date)
    end_ts = pd.Timestamp(end_date)
    pr = portfolio_returns[(portfolio_returns['date'] >= start_ts) & (portfolio_returns['date'] <= end_ts)].copy()
    mr = market_returns[(market_returns['date'] >= start_ts) & (market_returns['date'] <= end_ts)].copy()
    if pr.empty or mr.empty:
        return None, None
    pm_capm = calculate_all_portfolio_metrics(pr, mr, 'R&D Quintiles', rf_rate_monthly, model='capm')
    pm_ff3 = calculate_all_portfolio_metrics(pr, mr, 'R&D Quintiles', rf_rate_monthly, model='ff3')
    return pm_capm, pm_ff3

SUBPERIODS = [
    ('1980-2000', '1980-01-01', '2000-12-31'),
    ('2001-2019', '2001-01-01', '2019-12-31'),
]

portfolio_returns_norm, portfolio_metrics_capm_norm, portfolio_metrics_ff3_norm = run_quintile_metrics(analysis_data_norm, market_returns, rf_rate_monthly)
portfolio_returns_nonnorm, portfolio_metrics_capm_nonnorm, portfolio_metrics_ff3_nonnorm = run_quintile_metrics(analysis_data_nonnorm, market_returns, rf_rate_monthly)
portfolio_returns_revt, portfolio_metrics_capm_revt, portfolio_metrics_ff3_revt = run_quintile_metrics(analysis_data_revt, market_returns, rf_rate_monthly)
portfolio_returns_mkteq, portfolio_metrics_capm_mkteq, portfolio_metrics_ff3_mkteq = run_quintile_metrics(analysis_data_mkteq, market_returns, rf_rate_monthly)

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_18480/544992654.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_18480/544992654.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_18480/544992654.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping col

## Outputs: Performance Tables, Alpha Table, Chart

In [260]:
from IPython.display import HTML, display

TEST_PERIOD = '1980–2022'
colors = {'CRSP VW Benchmark': '#58508d', 'No R&D': '#003f5c', 'Q1': '#ffa600', 'Q2': '#dd5182', 'Q3': '#955196', 'Q4': '#bc5090', 'Q5': '#ff6361', 'Q5-No R&D': '#d80020', 'Q5-Q1': '#00c9a7'}
market_vw = market_returns.assign(date=market_returns['date'] + pd.offsets.MonthEnd(0), key='vw_CRSP_VW_Benchmark', value=market_returns['vwretd'])

def _show(period_label, pm_capm, pm_ff3, rd_label):
    display(HTML(create_alpha_table_html(create_alpha_table(pm_capm, market_returns, rf_rate_monthly), custom_css, caption_suffix=f'<br>{rd_label}<br>Test period: {period_label}', table_type='CAPM')))
    display(HTML(create_alpha_table_html(create_alpha_table(pm_ff3, market_returns, rf_rate_monthly), custom_css, caption_suffix=f'<br>{rd_label}<br>Test period: {period_label}', table_type='FF3')))

variants = [
    ('Normalized (XRDW % of NI)', portfolio_returns_norm, portfolio_metrics_capm_norm, portfolio_metrics_ff3_norm),
    ('Non-normalized (raw XRDW)', portfolio_returns_nonnorm, portfolio_metrics_capm_nonnorm, portfolio_metrics_ff3_nonnorm),
    ('Normalized (XRDW % of Revenue)', portfolio_returns_revt, portfolio_metrics_capm_revt, portfolio_metrics_ff3_revt),
    ('Normalized (XRDW % of Market Equity)', portfolio_returns_mkteq, portfolio_metrics_capm_mkteq, portfolio_metrics_ff3_mkteq),
]
returns_data_norm = None
returns_data_nonnorm = None
returns_data_revt = None
returns_data_mkteq = None

for rd_label, pr, pm_capm, pm_ff3 in variants:
    _show(TEST_PERIOD, pm_capm, pm_ff3, rd_label)
    for period_label, start_date, end_date in SUBPERIODS:
        pm_capm_sub, pm_ff3_sub = metrics_for_period(pr, market_returns, rf_rate_monthly, start_date, end_date)
        if pm_capm_sub is None or pm_capm_sub.empty:
            continue
        _show(period_label, pm_capm_sub, pm_ff3_sub, rd_label)

    chart_data = (pr.assign(key=pr['portfolio_id'].astype(str))
        .melt(id_vars=['date', 'portfolio_id', 'key'], value_vars=['ew_return', 'vw_return'], var_name='weighting', value_name='value')
        .assign(key=lambda x: x['weighting'].str.replace('_return', '') + '_' + x['portfolio_id'].astype(str)))
    chart_data = pd.concat([chart_data[['date', 'key', 'value']], market_vw[['date', 'key', 'value']]], ignore_index=True)
    t0 = chart_data.groupby('key')['date'].min().max()
    chart_data = chart_data[chart_data['date'] >= t0].sort_values(['key', 'date'])
    chart_data['cumvalue'] = (1 + pd.to_numeric(chart_data['value'], errors='coerce')).groupby(chart_data['key']).cumprod()
    chart_data['cumvalue'] = chart_data.groupby('key')['cumvalue'].transform(lambda x: x / x.iloc[0])
    fig = create_performance_chart(chart_data, panel_type='vw', colors=colors)
    fig.update_layout(title=f'Cumulative Performance (1980-2022) — {rd_label}')
    fig.show()

    ret_wide = pr.pivot(index='date', columns='portfolio_id', values='ew_return').add_suffix('_ew').join(
        pr.pivot(index='date', columns='portfolio_id', values='vw_return').add_suffix('_vw')).reset_index()
    if rd_label == 'Normalized (XRDW % of NI)':
        returns_data_norm = ret_wide
    elif rd_label == 'Non-normalized (raw XRDW)':
        returns_data_nonnorm = ret_wide
    elif rd_label == 'Normalized (XRDW % of Revenue)':
        returns_data_revt = ret_wide
    elif rd_label == 'Normalized (XRDW % of Market Equity)':
        returns_data_mkteq = ret_wide



Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
No R&D VW,-0.03,-0.69,0.489,0.99,0.96,0.47
Q1 VW,-0.02,-0.25,0.802,0.84,0.75,0.41
Q4 EW,0.29**,2.14,0.033,1.22,0.76,0.6
Q5 EW,0.41**,2.39,0.017,1.24,0.68,0.62
Q5-No R&D EW,0.07,0.6,0.551,0.05,0.01,0.14
Q5-Q1 EW,0.12,0.84,0.402,0.19,0.06,0.25
Q2 VW,0.10,0.91,0.362,1.06,0.78,0.51
Q3 VW,0.13,1.29,0.199,1.08,0.83,0.54
Q4 VW,0.01,0.12,0.904,1.16,0.8,0.46
Q5 VW,0.26*,1.74,0.082,1.24,0.73,0.57


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D VW,-0.08*,-1.92,0.056,0.98,0.13,0.06,0.96,0.47
Q1 VW,-0.09,-1.0,0.316,0.89,-0.22,0.1,0.77,0.41
Q4 EW,0.30***,3.67,0.000,1.08,0.85,-0.08,0.91,0.6
Q5 EW,0.44***,4.07,0.000,1.06,1.05,-0.1,0.87,0.62
Q5-No R&D EW,0.16,1.54,0.123,0.0,0.05,-0.3,0.16,0.14
Q5-Q1 EW,0.24**,2.02,0.044,0.06,0.49,-0.34,0.39,0.25
Q2 VW,0.11,1.01,0.313,1.04,-0.06,-0.17,0.79,0.51
Q3 VW,0.19**,2.17,0.031,1.03,0.04,-0.33,0.87,0.54
Q4 VW,0.05,0.47,0.636,1.1,0.12,-0.26,0.82,0.46
Q5 VW,0.34**,2.54,0.011,1.13,0.31,-0.37,0.79,0.57


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
No R&D VW,-0.05,-0.79,0.433,0.97,0.96,0.51
Q1 VW,-0.08,-0.56,0.577,0.89,0.76,0.43
Q4 EW,0.10,0.45,0.656,1.21,0.69,0.52
Q5 EW,0.28,0.95,0.342,1.21,0.59,0.57
Q5-No R&D EW,-0.16,-1.0,0.319,0.21,0.12,-0.02
Q5-Q1 EW,-0.05,-0.22,0.827,0.21,0.06,0.09
Q2 VW,-0.08,-0.53,0.596,1.08,0.81,0.45
Q3 VW,0.23,1.56,0.121,1.09,0.81,0.66
Q4 VW,-0.14,-0.79,0.432,1.11,0.77,0.41
Q5 VW,0.26,1.24,0.215,1.19,0.73,0.62


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D VW,-0.11*,-1.94,0.054,0.97,0.12,0.08,0.96,0.51
Q1 VW,-0.20,-1.49,0.137,0.94,-0.21,0.09,0.79,0.43
Q4 EW,0.28**,2.13,0.034,1.05,0.89,-0.09,0.91,0.52
Q5 EW,0.53***,3.15,0.002,1.0,1.12,-0.14,0.87,0.57
Q5-No R&D EW,0.09,0.59,0.558,0.06,0.21,-0.33,0.35,-0.02
Q5-Q1 EW,0.31*,1.71,0.089,-0.01,0.56,-0.41,0.48,0.09
Q2 VW,-0.01,-0.05,0.956,1.02,-0.17,-0.21,0.82,0.45
Q3 VW,0.48***,3.52,0.001,0.93,0.05,-0.41,0.86,0.66
Q4 VW,0.06,0.35,0.724,0.98,0.08,-0.33,0.81,0.41
Q5 VW,0.47**,2.57,0.011,1.03,0.39,-0.28,0.8,0.62


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
No R&D VW,-0.00,-0.0,0.996,1.0,0.96,0.43
Q1 VW,0.09,0.69,0.492,0.77,0.74,0.46
Q4 EW,0.43***,2.64,0.009,1.27,0.83,0.65
Q5 EW,0.61***,2.91,0.004,1.31,0.77,0.71
Q5-No R&D EW,0.27**,2.05,0.042,-0.03,0.0,0.45
Q5-Q1 EW,0.24,1.31,0.191,0.21,0.1,0.42
Q2 VW,0.26,1.44,0.152,1.05,0.73,0.55
Q3 VW,-0.00,-0.02,0.981,1.1,0.86,0.4
Q4 VW,0.07,0.42,0.672,1.26,0.82,0.44
Q5 VW,0.26,1.18,0.239,1.36,0.76,0.51


...

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D VW,-0.06,-1.18,0.238,0.94,0.0,0.15,0.92,0.49
Q1 VW,0.47***,5.24,0.000,0.99,-0.05,-0.31,0.85,0.73
Q4 EW,-0.01,-0.05,0.964,1.12,1.15,-0.05,0.83,0.39
Q5 EW,-2.42***,-10.78,0.000,1.16,1.33,0.0,0.67,-0.6
Q5-No R&D EW,-2.66***,-14.41,0.000,0.14,0.56,-0.41,0.28,-1.86
Q5-Q1 EW,-4.23***,-19.9,0.000,0.13,0.32,0.16,0.06,-2.89
Q2 VW,0.26***,2.74,0.006,1.03,0.02,-0.27,0.85,0.59
Q3 VW,-0.12,-1.21,0.226,1.07,0.2,-0.12,0.84,0.38
Q4 VW,-0.82***,-5.81,0.000,1.15,0.21,-0.01,0.75,-0.0
Q5 VW,-1.87***,-10.34,0.000,1.33,0.45,0.38,0.71,-0.39


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
No R&D VW,-0.01,-0.06,0.949,0.92,0.91,0.53
Q1 VW,0.44***,2.78,0.006,1.14,0.81,0.77
Q4 EW,-0.30,-1.05,0.296,1.17,0.58,0.28
Q5 EW,-2.27***,-6.19,0.000,1.11,0.43,-0.67
Q5-No R&D EW,-2.67***,-10.4,0.000,0.2,0.05,-2.13
Q5-Q1 EW,-4.22***,-16.93,0.000,-0.15,0.03,-3.8
Q2 VW,0.12,0.79,0.431,1.02,0.8,0.58
Q3 VW,-0.02,-0.11,0.910,1.02,0.8,0.49
Q4 VW,-0.77***,-4.28,0.000,1.07,0.75,0.01
Q5 VW,-1.19***,-5.35,0.000,1.08,0.66,-0.24


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D VW,-0.15*,-1.91,0.057,0.98,-0.01,0.17,0.92,0.53
Q1 VW,0.68***,4.65,0.000,0.98,-0.08,-0.45,0.85,0.77
Q4 EW,-0.12,-0.64,0.525,1.0,1.08,-0.04,0.84,0.28
Q5 EW,-2.10***,-7.6,0.000,0.95,1.23,0.02,0.7,-0.67
Q5-No R&D EW,-2.36***,-10.82,0.000,0.01,0.55,-0.33,0.36,-2.13
Q5-Q1 EW,-4.32***,-16.89,0.000,-0.1,0.14,0.2,0.05,-3.8
Q2 VW,0.25*,1.71,0.088,0.92,0.02,-0.25,0.81,0.58
Q3 VW,0.00,0.0,0.996,0.98,0.25,-0.01,0.83,0.49
Q4 VW,-0.73***,-3.98,0.000,1.02,0.05,-0.1,0.75,0.01
Q5 VW,-1.39***,-6.18,0.000,1.16,0.19,0.31,0.68,-0.24


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
No R&D VW,0.07,0.87,0.383,0.91,0.92,0.47
Q1 VW,0.34***,2.95,0.004,0.89,0.84,0.68
Q4 EW,0.36,1.26,0.210,1.49,0.7,0.52
Q5 EW,-2.30***,-5.72,0.000,1.7,0.59,-0.51
Q5-No R&D EW,-2.69***,-8.51,0.000,0.51,0.18,-1.61
Q5-Q1 EW,-3.81***,-11.5,0.000,0.51,0.16,-2.26
Q2 VW,0.24*,1.7,0.091,1.16,0.85,0.56
Q3 VW,-0.17,-1.16,0.248,1.25,0.85,0.3
Q4 VW,-0.71***,-3.21,0.002,1.44,0.78,0.03
Q5 VW,-1.91***,-6.65,0.000,1.67,0.74,-0.42


Portfolio,Alpha (monthly %),t-statistic,p-value,Beta (Mkt),Beta (SMB),Beta (HML),R²,Sharpe
No R&D VW,0.02,0.3,0.766,0.89,0.07,0.09,0.92,0.47
Q1 VW,0.33***,3.19,0.002,0.92,-0.09,-0.28,0.87,0.68
Q4 EW,0.18,0.84,0.402,1.29,1.17,-0.24,0.82,0.52
Q5 EW,-2.50***,-7.05,0.000,1.47,1.29,-0.22,0.69,-0.51
Q5-No R&D EW,-2.73***,-9.32,0.000,0.45,0.46,-0.57,0.3,-1.61
Q5-Q1 EW,-3.88***,-11.9,0.000,0.43,0.46,-0.1,0.2,-2.26
Q2 VW,0.21*,1.72,0.086,1.17,0.09,-0.4,0.89,0.56
Q3 VW,-0.23*,-1.68,0.095,1.25,0.15,-0.27,0.88,0.3
Q4 VW,-0.81***,-3.83,0.000,1.38,0.45,-0.16,0.8,0.03
Q5 VW,-2.05***,-7.63,0.000,1.55,0.66,0.22,0.77,-0.42
